# How to Fix Class Imbalance

<a href="https://colab.research.google.com/github/quprep/quprep/blob/main/examples/how-to/fix_class_imbalance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/quprep/quprep/v0.10.0?labpath=examples%2Fhow-to%2Ffix_class_imbalance.ipynb)
<a href="https://account.qbraid.com?gitHubUrl=https://github.com/quprep/quprep/blob/main/examples/how-to/fix_class_imbalance.ipynb"><img src="https://qbraid-static.s3.amazonaws.com/logos/Launch_on_qBraid_white.png" height="23"></a>


`ImbalanceHandler` corrects skewed class distributions before encoding. Three strategies: `smote` (synthetic minority oversampling), `oversample` (random duplication), and `undersample` (majority class reduction).

In [ ]:
!pip install -q quprep

In [ ]:
import warnings

import numpy as np

import quprep as qd
from quprep import QuPrepWarning

print(f"quprep {qd.__version__}")

rng = np.random.default_rng(0)
X = rng.uniform(0, np.pi, (100, 4))
y = np.array([0] * 90 + [1] * 10)  # 9:1 imbalance
dataset = qd.NumpyIngester().load(X, y=y)

print(f"Before: class counts = {np.bincount(y)}")

## 1. SMOTE — synthetic minority oversampling

In [ ]:
smote_ds = qd.ImbalanceHandler(strategy="smote").fit_transform(dataset)
print(f"SMOTE      → {smote_ds.data.shape[0]} samples")
print(f"            class counts = {np.bincount(smote_ds.labels.astype(int))}")

## 2. Random oversample

In [ ]:
over_ds = qd.ImbalanceHandler(strategy="oversample").fit_transform(dataset)
print(f"Oversample → {over_ds.data.shape[0]} samples")
print(f"            class counts = {np.bincount(over_ds.labels.astype(int))}")

## 3. Undersample

In [ ]:
under_ds = qd.ImbalanceHandler(strategy="undersample").fit_transform(dataset)
print(f"Undersample → {under_ds.data.shape[0]} samples")
print(f"             class counts = {np.bincount(under_ds.labels.astype(int))}")

## 4. Pipeline: balance → normalise → encode

In [ ]:
balanced_ds = qd.ImbalanceHandler(strategy="smote").fit_transform(dataset)

with warnings.catch_warnings():
    warnings.simplefilter("ignore", QuPrepWarning)
    result = qd.Pipeline(
        normalizer=qd.Scaler(strategy="minmax_pi"),
        encoder=qd.AngleEncoder(),
    ).fit_transform(balanced_ds)

print(f"Circuits : {len(result.encoded)}")
print(f"Qubits   : {result.encoded[0].metadata['n_qubits']}")